 ## DEM of Difference (DoD) & Volume of Erosion/Deposition
### Compares two DEMs (e.g. two survey epochs) and computes:
### - DoD = DEM_after - DEM_before
### - Erosion volume (negative change) and Deposition volume (positive change)
### A **Level of Detection (LoD)** threshold masks out changes below survey noise.

#### CRS: EPSG:25832.


In [ ]:
# USER CONFIGURATION

DEM_BEFORE_PATH = "data/dem_epoch1.tif"   # earlier survey
DEM_AFTER_PATH  = "data/dem_epoch2.tif"   # later survey
OUTPUT_DOD_PATH = "output/dod.tif"

# Level of Detection: ignore change below this value (metres).
# Set from your DEM vertical uncertainty; use 0.0 to disable.
LOD_THRESHOLD   = 0.15

In [ ]:
# Import required Libraries

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from dem_utils import read_dem, write_raster

In [ ]:
# Load and align DEMs

# The 'after' DEM is resampled onto the 'before' DEM grid so the pixels match.

with rasterio.open(DEM_BEFORE_PATH) as ref:
    before = ref.read(1).astype("float64")
    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_nodata = ref.nodata
    res_x, res_y = ref.res

if ref_nodata is not None:
    before[before == ref_nodata] = np.nan

# Reproject/resample 'after' to the reference grid
after = np.empty_like(before)
with rasterio.open(DEM_AFTER_PATH) as src:
    reproject(
        source=rasterio.band(src, 1),
        destination=after,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.bilinear,
        dst_nodata=np.nan,
    )

pixel_area = abs(res_x * res_y)   # m^2 per cell
print(f"Pixel size: {res_x} x {res_y} m  ->  cell area = {pixel_area:.4f} m^2")

In [ ]:
# Compute DEM of Difference (DoD)

dod = after - before   # positive = gain (deposition), negative = loss (erosion)

# Apply Level of Detection threshold
if LOD_THRESHOLD > 0:
    dod_thresh = np.where(np.abs(dod) < LOD_THRESHOLD, np.nan, dod)
else:
    dod_thresh = dod.copy()

In [ ]:
# Calculate Volume of deposition and erosion

valid = ~np.isnan(dod_thresh)

deposition = np.where(dod_thresh > 0, dod_thresh, 0.0)
erosion    = np.where(dod_thresh < 0, dod_thresh, 0.0)

vol_deposition = np.nansum(deposition) * pixel_area          # m^3 (+)
vol_erosion    = np.nansum(erosion) * pixel_area             # m^3 (-)
net_volume     = vol_deposition + vol_erosion                # m^3

area_deposition = np.sum(dod_thresh > 0) * pixel_area        # m^2
area_erosion    = np.sum(dod_thresh < 0) * pixel_area        # m^2

print("=== VOLUME CHANGE REPORT ===")
print(f"Level of Detection applied : {LOD_THRESHOLD} m")
print(f"Deposition volume : {vol_deposition:,.2f} m^3  over {area_deposition:,.1f} m^2")
print(f"Erosion volume    : {vol_erosion:,.2f} m^3  over {area_erosion:,.1f} m^2")
print(f"Net volume change : {net_volume:,.2f} m^3")

In [ ]:
# Visualize the DoD and save to output file

vmax = np.nanpercentile(np.abs(dod_thresh), 98)
plt.figure(figsize=(10, 8))
im = plt.imshow(dod_thresh, cmap="RdBu", vmin=-vmax, vmax=vmax)
plt.colorbar(im, label="Elevation change [m]")
plt.title("DEM of Difference (red = erosion, blue = deposition)")
plt.axis("off")
plt.show()

In [ ]:
# Save DoD raster

write_raster(OUTPUT_DOD_PATH, dod_thresh, ref_profile)